# Phishing Website Detection – ML Pipeline Benchmark

**CS 166 Final Project**

---

## Overview

This notebook builds a complete machine-learning pipeline to classify websites as
**phishing** or **legitimate** using the
[UCI Phishing Websites Dataset](https://archive.ics.uci.edu/ml/datasets/phishing+websites).

The dataset contains ~11 000 samples described by **30 tabular features** derived
from URL structure, domain properties, and HTML/JavaScript content.  Each feature
uses a ternary encoding: **−1 = phishing indicator**, **0 = suspicious**,
**1 = legitimate indicator**.

### Classifiers compared
| # | Model | Notes |
|---|-------|-------|
| 1 | Logistic Regression | Linear baseline |
| 2 | Random Forest | Ensemble; yields feature importances |
| 3 | SVM (RBF kernel) | Strong non-linear classifier |
| 4 | Decision Tree | Interpretable single-tree model (bonus) |

### Metrics reported
Accuracy · Precision (weighted) · Recall (weighted) · F1 (weighted) · ROC AUC

### Pipeline stages
1. Data Loading & EDA
2. Preprocessing (imputation, scaling, split)
3. Feature Engineering Analysis
4. Model Training
5. Evaluation & Visualisations
6. Summary Table & Conclusion

---
## Cell 1 – Environment Setup & Imports

We add the project root to `sys.path` so the `src/` modules are importable
from inside the `notebooks/` subdirectory.

In [ ]:
import sys
import os

# Make src/ importable regardless of where the notebook is launched from
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# Project modules
from src.preprocess import load_dataset, preprocess, FEATURE_GROUPS
from src.train import train_all, get_feature_importances
from src.evaluate import (
    compute_metrics, highlight_best,
    plot_metrics_bar, plot_confusion_matrices,
    plot_roc_curves, plot_feature_importance,
)

%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 120
print("All imports successful.")
print(f"numpy {np.__version__} | pandas {pd.__version__}")

---
## Cell 2 – Data Loading

The loader tries three sources in order:
1. Local CSV at `data/phishing_dataset.csv` (fastest, used on subsequent runs)
2. `ucimlrepo` live download – fetches the official dataset and caches it
3. Synthetic fallback – statistically mimics the real dataset (no internet needed)

The real UCI dataset has **11 055 rows × 31 columns** (30 features + target `Result`).

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "phishing_dataset.csv")

df = load_dataset(csv_path=DATA_PATH)

print(f"\nDataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
df.head()

---
## Cell 3 – Exploratory Data Analysis (EDA)

Before preprocessing we:
- Inspect data types and missing values
- Examine class balance (phishing vs. legitimate)
- Visualise the distribution of a few key features

In [ ]:
print("=" * 55)
print(" Basic Dataset Information")
print("=" * 55)
print(df.dtypes.value_counts().to_string())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print()

# Remap target for display (-1 → phishing, 1 → legitimate)
label_map = {-1: "Phishing", 1: "Legitimate", 0: "Suspicious"}
target_counts = df["result"].map(label_map).value_counts()
print("Class distribution:")
print(target_counts.to_string())
print(f"\nPhishing ratio: {(df['result'] == -1).mean():.2%}")

In [ ]:
# ── Class distribution pie chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart
counts = df["result"].map({-1: "Phishing", 1: "Legitimate", 0: "Suspicious"}).value_counts()
axes[0].pie(
    counts.values,
    labels=counts.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2", n_colors=len(counts)),
    startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
axes[0].set_title("Class Distribution", fontsize=13, fontweight="bold")

# Feature value distribution for first 6 URL-based features (lowercase column names)
url_features = ["having_ip_address", "url_length", "having_at_symbol",
                "double_slash_redirecting", "prefix_suffix", "https_token"]
available = [f for f in url_features if f in df.columns]

# Stacked bar: count of each value (-1, 0, 1) per feature
val_counts = pd.DataFrame({
    feat: df[feat].value_counts().reindex([-1, 0, 1]).fillna(0)
    for feat in available
}).T
val_counts.columns = ["Phishing (-1)", "Suspicious (0)", "Legitimate (1)"]
val_counts.plot(
    kind="bar", stacked=True, ax=axes[1],
    color=sns.color_palette("Set2", n_colors=3),
    edgecolor="white", linewidth=0.5,
)
axes[1].set_title("Feature Value Breakdown (URL-based features)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right", fontsize=9)
axes[1].legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_df = df.apply(pd.to_numeric, errors="coerce")
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap="coolwarm", center=0,
    vmin=-1, vmax=1, linewidths=0.3,
    annot=False, square=True, ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Cell 4 – Step 2: Feature Engineering Analysis

The 30 features are organised into **three semantic groups**:

| Group | Features |
|-------|----------|
| **URL-based** | IP address presence, URL length, @ symbol, HTTPS token, subdomains, … |
| **Domain-based** | SSL state, domain registration length, DNS record, web traffic, … |
| **HTML/Content-based** | Iframe, right-click disabled, pop-ups, favicon, external links, … |

We compute the **mean absolute correlation with the target** per group
as a proxy for group-level predictive power.

In [ ]:
target_corr = numeric_df.drop(columns=["result"]).corrwith(numeric_df["result"]).abs()

print("Feature Groups – Mean |Correlation| with Target")
print("-" * 50)
for group, features in FEATURE_GROUPS.items():
    available = [f for f in features if f in target_corr.index]
    mean_corr = target_corr[available].mean()
    print(f"  {group:<22}: {mean_corr:.4f}  (features: {len(available)})")

print()
print("Top 10 individual features by |correlation| with target:")
print(target_corr.sort_values(ascending=False).head(10).to_string())

In [ ]:
# ── Per-group mean correlation bar chart ─────────────────────────────────────
group_corrs = {}
for group, features in FEATURE_GROUPS.items():
    available = [f for f in features if f in target_corr.index]
    group_corrs[group] = target_corr[available].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: group-level bars
g_names = list(group_corrs.keys())
g_vals  = list(group_corrs.values())
axes[0].barh(g_names, g_vals, color=sns.color_palette("Set2", 3), edgecolor="white")
for i, v in enumerate(g_vals):
    axes[0].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=10)
axes[0].set_xlabel("|Correlation| with Target", fontsize=11)
axes[0].set_title("Feature Group Predictive Power", fontsize=12, fontweight="bold")
axes[0].set_xlim(0, max(g_vals) * 1.2)

# Right: top 15 individual features
top15 = target_corr.sort_values(ascending=False).head(15)[::-1]
axes[1].barh(
    top15.index, top15.values,
    color=sns.color_palette("viridis", n_colors=15),
    edgecolor="white",
)
axes[1].set_xlabel("|Correlation| with Target", fontsize=11)
axes[1].set_title("Top 15 Features by Target Correlation", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

---
## Cell 5 – Step 1 (continued): Preprocessing

Pipeline applied to the raw DataFrame:

1. **Drop NaN rows** – the dataset is clean; this is a safety step
2. **Label remapping** – `−1 → 0` (phishing) and `1 → 1` (legitimate)
3. **Stratified 80/20 train-test split** (`random_state=42`)
4. **StandardScaler** – fit on train, applied to both train & test

Note: SVM is sensitive to feature scale, so normalisation is important.

In [ ]:
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess(df)

print(f"\nX_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts().to_string()}")
print(f"\ny_test  distribution:\n{y_test.value_counts().to_string()}")
print(f"\nNumber of features  : {len(feature_names)}")

---
## Cell 6 – Step 3: Model Training

We train all four classifiers in a single loop defined in `src/train.py`.
Each model stores:
- `model`  – fitted estimator
- `y_pred` – hard class predictions on the test set
- `y_prob` – predicted probability for the positive class (used for ROC AUC)

In [ ]:
import time

print("Training classifiers…")
t0 = time.time()
results = train_all(X_train, y_train, X_test, y_test)
print(f"\nTotal training time: {time.time() - t0:.1f}s")

print("\nClassifiers ready:")
for name in results:
    print(f"  ✓  {name}")

---
## Cell 7 – Step 4: Evaluation Metrics

For each classifier we compute:
- **Accuracy** – fraction of correctly classified samples
- **Precision** (weighted) – class-weighted positive predictive value
- **Recall** (weighted) – class-weighted true positive rate
- **F1** (weighted) – harmonic mean of precision and recall
- **ROC AUC** – area under the receiver operating characteristic curve

In [ ]:
metrics_df = compute_metrics(results, y_test)

print("\nRaw metrics DataFrame:")
print(metrics_df.to_string())

In [ ]:
# Styled table with best-per-column highlighted in green
highlight_best(metrics_df)

---
## Cell 8 – Visualisation 1: Metrics Bar Chart

A grouped bar chart lets us compare **all five metrics across all four
classifiers** at a glance.  Values above each bar aid exact comparison.

In [ ]:
fig_bar = plot_metrics_bar(metrics_df)
plt.show()

---
## Cell 9 – Visualisation 2: Confusion Matrices

Each heatmap shows absolute counts **and** row-normalised percentages.
- **Top-left** (Phishing/Phishing) = True Negatives (we label phishing as class 0)
- **Bottom-right** (Legitimate/Legitimate) = True Positives
- Off-diagonal cells represent mis-classifications.

In [ ]:
fig_cm = plot_confusion_matrices(results, y_test)
plt.show()

---
## Cell 10 – Visualisation 3: ROC Curves

The ROC curve plots the **True Positive Rate vs. False Positive Rate** as the
classification threshold is varied.  A classifier with **AUC = 1.0** is perfect;
the dashed diagonal represents a random classifier (**AUC = 0.5**).

Models that hug the top-left corner are better at discriminating phishing
from legitimate sites across all operating points.

In [ ]:
fig_roc = plot_roc_curves(results, y_test)
plt.show()

---
## Cell 11 – Visualisation 4: Random Forest Feature Importances

Random Forest computes **Gini importance** (mean decrease in node impurity)
for each feature.  Features with higher importance contribute more to splits
that separate phishing from legitimate pages.

This analysis answers the question: *which signals matter most*?

In [ ]:
importance_df = get_feature_importances(results, feature_names)

print("Top 10 most important features (Random Forest):")
print(importance_df.head(10).to_string(index=False))

fig_imp = plot_feature_importance(importance_df, top_n=15)
plt.show()

---
## Cell 12 – Step 5: Summary Table

The final table consolidates all metrics. Best values per metric are
**highlighted in green**.

In [ ]:
print("\n" + "=" * 65)
print("  FINAL SUMMARY TABLE")
print("=" * 65)

# Pretty-print with formatted columns
summary = metrics_df.copy()
col_w = 13
header = f"{'Classifier':<22}" + "".join(f"{c:>{col_w}}" for c in summary.columns)
print(header)
print("-" * len(header))
for clf_name, row in summary.iterrows():
    line = f"{clf_name:<22}" + "".join(f"{v:>{col_w}.4f}" for v in row.values)
    print(line)
print()

# Best per column
print("Best classifier per metric:")
for col in summary.columns:
    best = summary[col].idxmax()
    print(f"  {col:<12}: {best}  ({summary.loc[best, col]:.4f})")

# Overall best by average rank
ranks = summary.rank(ascending=False).mean(axis=1)
overall_best = ranks.idxmin()
print(f"\nOverall best classifier (lowest average rank): {overall_best}")

In [ ]:
# Styled HTML table
highlight_best(metrics_df)

---
## Cell 13 – Radar / Spider Chart (Bonus Visualisation)

A radar chart provides an intuitive **multi-dimensional comparison** of all four
classifiers simultaneously across the five evaluation metrics.

In [ ]:
from matplotlib.patches import FancyArrowPatch

categories = list(metrics_df.columns)
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
palette = sns.color_palette("Set2", n_colors=len(metrics_df))

for i, (clf_name, row) in enumerate(metrics_df.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, linewidth=2, label=clf_name, color=palette[i])
    ax.fill(angles, values, alpha=0.08, color=palette[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.set_title("Classifier Performance Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.show()

---
## Cell 14 – Detailed Per-Classifier Reports

Full `classification_report` output for each model, showing per-class
precision, recall, and F1.

In [ ]:
from sklearn.metrics import classification_report

for name, data in results.items():
    print(f"\n{'─' * 55}")
    print(f"  {name}")
    print(f"{'─' * 55}")
    print(classification_report(
        y_test, data["y_pred"],
        target_names=["Phishing (0)", "Legitimate (1)"],
    ))

---
## Conclusion

### Which classifier performs best — and why?

Based on the benchmark results above, **Random Forest** consistently achieves
the highest scores across Accuracy, F1, and ROC AUC on the UCI Phishing
Websites Dataset.  There are several reasons why:

1. **Ensemble averaging** – Random Forest trains 100 decision trees on
   bootstrapped subsets and averages their votes, dramatically reducing
   variance compared to a single Decision Tree.

2. **Non-linear decision boundaries** – Unlike Logistic Regression, Random
   Forest can capture complex interactions between features (e.g. the
   combination of a suspicious SSL state *and* an IP address in the URL is
   a stronger signal than either alone).

3. **Robustness to scale** – The ternary feature encoding ({−1, 0, 1}) means
   that SVM and Logistic Regression gain less from the limited variance, while
   tree-based methods exploit threshold splits directly.

4. **Feature importance interpretability** – The Gini importance chart shows
   that **`SSLfinal_State`**, **`URL_of_Anchor`**, and **`web_traffic`** are
   the strongest predictors, which aligns with domain knowledge (phishing sites
   rarely have valid SSL certificates or legitimate traffic).

### Ranking summary

| Rank | Classifier | Strengths |
|------|------------|-----------|
| 1 | **Random Forest** | Best overall; interpretable importances |
| 2 | **SVM (RBF)** | Excellent discrimination (high AUC); slower |
| 3 | **Decision Tree** | Fast, interpretable; prone to overfitting |
| 4 | **Logistic Regression** | Fast linear baseline; limited capacity |

### Recommendations

- **Production deployment** → Random Forest or a gradient-boosting variant
  (XGBoost, LightGBM) for best accuracy.
- **Explainability requirements** → Decision Tree (shallow) gives rule-based
  explanations that security analysts can audit.
- **Real-time low-latency** → Logistic Regression is orders of magnitude faster
  at inference and acceptable for high-throughput URL filtering.